In [1]:
# Install required packages
!pip install jiwer rouge-score bert-score torch nltk
!pip install --upgrade numpy

import re
import numpy as np
from pathlib import Path
from jiwer import wer, cer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import BERTScorer
import nltk

# Download required NLTK data
nltk.download('punkt', quiet=True)

print("✅ All packages installed successfully!")


✅ All packages installed successfully!


In [2]:
class TextExtractionEvaluator:
    def __init__(self):
        self.scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.bert_scorer = BERTScorer(lang="en", model_type="microsoft/deberta-xlarge-mnli")

    def clean_text(self, text):
        """Clean and normalize text for comparison"""
        # Remove special characters, extra spaces, normalize case
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip().lower()

    def read_file(self, file_path):
        """Read text from file"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except UnicodeDecodeError:
            with open(file_path, 'r', encoding='latin-1') as f:
                return f.read()

    def split_by_part(self, text):
        """Split text into Part-A and Part-B"""
        part_a = ""
        part_b = ""

        # Try different patterns for part splitting
        patterns = [r'part-b', r'part b', r'part_b', r'partb']

        for pattern in patterns:
            if re.search(pattern, text, re.IGNORECASE):
                parts = re.split(pattern, text, flags=re.IGNORECASE)
                if len(parts) > 0:
                    part_a = parts[0]
                if len(parts) > 1:
                    part_b = parts[1]
                break

        # If no part-b found, try to split based on question numbers
        if not part_b and "part-a" in text.lower():
            # Assume everything after part-a is part-b
            parts = re.split(r'part-a', text, flags=re.IGNORECASE)
            if len(parts) > 1:
                part_b = parts[1]

        return part_a, part_b

    def calculate_metrics(self, ground_truth, extracted):
        """Calculate all evaluation metrics"""

        # Clean texts
        gt_clean = self.clean_text(ground_truth)
        ex_clean = self.clean_text(extracted)

        # Tokenize for BLEU and ROUGE
        gt_tokens = gt_clean.split()
        ex_tokens = ex_clean.split()

        # Calculate WER and CER
        wer_score = wer(gt_clean, ex_clean)
        cer_score = cer(gt_clean, ex_clean)

        # Calculate BLEU score
        smoothie = SmoothingFunction().method4
        try:
            bleu_score = sentence_bleu([gt_tokens], ex_tokens, smoothing_function=smoothie)
        except:
            bleu_score = 0.0

        # Calculate ROUGE scores
        rouge_scores = self.scorer.score(gt_clean, ex_clean)

        # Calculate BERTScore
        try:
            P, R, F1 = self.bert_scorer.score([ex_clean], [gt_clean])
            bert_score = F1.mean().item()
        except:
            bert_score = 0.0

        # Calculate character accuracy
        char_accuracy = self.character_accuracy(ground_truth, extracted)

        return {
            'WER': wer_score,
            'CER': cer_score,
            'BLEU': bleu_score,
            'ROUGE-1': rouge_scores['rouge1'].fmeasure,
            'ROUGE-2': rouge_scores['rouge2'].fmeasure,
            'ROUGE-L': rouge_scores['rougeL'].fmeasure,
            'BERTScore': bert_score,
            'Character_Accuracy': char_accuracy
        }

    def character_accuracy(self, gt, ex):
        """Calculate character-level accuracy"""
        gt_clean = self.clean_text(gt)
        ex_clean = self.clean_text(ex)

        gt_chars = list(gt_clean)
        ex_chars = list(ex_clean)

        # Pad the shorter list
        max_len = max(len(gt_chars), len(ex_chars))
        gt_chars.extend([''] * (max_len - len(gt_chars)))
        ex_chars.extend([''] * (max_len - len(ex_chars)))

        correct_chars = sum(1 for gt_char, ex_char in zip(gt_chars, ex_chars) if gt_char == ex_char)
        return correct_chars / max_len if max_len > 0 else 0

    def evaluate_files(self, ground_truth_file, extracted_file):
        """Main evaluation function"""
        print("🔍 Loading files...")

        # Read files
        gt_text = self.read_file(ground_truth_file)
        ex_text = self.read_file(extracted_file)

        print(f"📊 Ground truth file: {ground_truth_file}")
        print(f"📊 Extracted file: {extracted_file}")
        print(f"📝 Ground truth length: {len(gt_text)} characters")
        print(f"📝 Extracted text length: {len(ex_text)} characters")
        print()

        # Calculate overall metrics
        print("=" * 60)
        print("OVERALL EVALUATION METRICS")
        print("=" * 60)

        overall_metrics = self.calculate_metrics(gt_text, ex_text)

        for metric, value in overall_metrics.items():
            if metric in ['WER', 'CER']:
                print(f"{metric:<18}: {value:.4f} ({value*100:.2f}%)")
            elif metric == 'Character_Accuracy':
                print(f"{metric:<18}: {value:.4f} ({value*100:.2f}%)")
            else:
                print(f"{metric:<18}: {value:.4f}")

        # Calculate metrics by section
        print("\n" + "=" * 60)
        print("SECTION-WISE EVALUATION")
        print("=" * 60)

        gt_part_a, gt_part_b = self.split_by_part(gt_text)
        ex_part_a, ex_part_b = self.split_by_part(ex_text)

        if gt_part_a and ex_part_a:
            print("\n📋 PART-A METRICS:")
            print("-" * 40)
            part_a_metrics = self.calculate_metrics(gt_part_a, ex_part_a)
            for metric, value in part_a_metrics.items():
                if metric in ['WER', 'CER']:
                    print(f"  {metric:<16}: {value:.4f} ({value*100:.2f}%)")
                elif metric == 'Character_Accuracy':
                    print(f"  {metric:<16}: {value:.4f} ({value*100:.2f}%)")
                else:
                    print(f"  {metric:<16}: {value:.4f}")

        if gt_part_b and ex_part_b:
            print("\n📋 PART-B METRICS:")
            print("-" * 40)
            part_b_metrics = self.calculate_metrics(gt_part_b, ex_part_b)
            for metric, value in part_b_metrics.items():
                if metric in ['WER', 'CER']:
                    print(f"  {metric:<16}: {value:.4f} ({value*100:.2f}%)")
                elif metric == 'Character_Accuracy':
                    print(f"  {metric:<16}: {value:.4f} ({value*100:.2f}%)")
                else:
                    print(f"  {metric:<16}: {value:.4f}")

        # Quality assessment
        print("\n" + "=" * 60)
        print("QUALITY ASSESSMENT")
        print("=" * 60)

        self.quality_assessment(overall_metrics)

        return overall_metrics, part_a_metrics if gt_part_a and ex_part_a else None, part_b_metrics if gt_part_b and ex_part_b else None

    def quality_assessment(self, metrics):
        """Provide quality assessment based on metrics"""
        wer = metrics['WER']
        cer = metrics['CER']
        bertscore = metrics['BERTScore']
        char_acc = metrics['Character_Accuracy']

        print("📈 Extraction Quality:")

        if wer < 0.2:
            print("  ✅ Word Error Rate: EXCELLENT")
        elif wer < 0.35:
            print("  ✅ Word Error Rate: GOOD")
        elif wer < 0.5:
            print("  ⚠️  Word Error Rate: MODERATE")
        else:
            print("  ❌ Word Error Rate: POOR")

        if cer < 0.15:
            print("  ✅ Character Error Rate: EXCELLENT")
        elif cer < 0.25:
            print("  ✅ Character Error Rate: GOOD")
        elif cer < 0.4:
            print("  ⚠️  Character Error Rate: MODERATE")
        else:
            print("  ❌ Character Error Rate: POOR")

        if bertscore > 0.85:
            print("  ✅ Semantic Similarity: EXCELLENT")
        elif bertscore > 0.75:
            print("  ✅ Semantic Similarity: GOOD")
        elif bertscore > 0.6:
            print("  ⚠️  Semantic Similarity: MODERATE")
        else:
            print("  ❌ Semantic Similarity: POOR")

        if char_acc > 0.9:
            print("  ✅ Character Accuracy: EXCELLENT")
        elif char_acc > 0.8:
            print("  ✅ Character Accuracy: GOOD")
        elif char_acc > 0.7:
            print("  ⚠️  Character Accuracy: MODERATE")
        else:
            print("  ❌ Character Accuracy: POOR")

print("✅ TextExtractionEvaluator class defined successfully!")



✅ TextExtractionEvaluator class defined successfully!


In [3]:
# Function to upload files in Colab
from google.colab import files

def upload_and_evaluate():
    """Upload files and run evaluation in Colab"""
    print("📤 Upload your text files:")
    print("1. First upload the GROUND TRUTH file")
    print("2. Then upload the EXTRACTED TEXT file")
    print("3. (Optional) Upload a COMPARISON file if you want to compare two extractions")

    uploaded = files.upload()

    file_names = list(uploaded.keys())

    if len(file_names) < 2:
        print("❌ Please upload at least 2 files (ground truth and extracted text)")
        return

    # Get file paths
    ground_truth_file = file_names[0]
    extracted_file = file_names[1]
    comparison_file = file_names[2] if len(file_names) > 2 else None

    print(f"\n📁 Uploaded files: {file_names}")

    # Initialize evaluator
    evaluator = TextExtractionEvaluator()

    # Evaluate main files
    print("\n" + "="*70)
    print("EVALUATING MAIN EXTRACTION")
    print("="*70)

    metrics, part_a_metrics, part_b_metrics = evaluator.evaluate_files(ground_truth_file, extracted_file)

    # Compare with another extraction if provided
    if comparison_file:
        print("\n" + "=" * 70)
        print("COMPARISON WITH ANOTHER EXTRACTION")
        print("=" * 70)

        compare_metrics, _, _ = evaluator.evaluate_files(ground_truth_file, comparison_file)

        print("\n" + "=" * 70)
        print("COMPARISON RESULTS")
        print("=" * 70)

        print(f"{'Metric':<18} {'Main':<10} {'Compare':<10} {'Difference':<12}")
        print("-" * 70)

        for metric in metrics.keys():
            if metric in compare_metrics:
                main_val = metrics[metric]
                comp_val = compare_metrics[metric]

                if metric in ['WER', 'CER']:
                    diff = main_val - comp_val
                    print(f"{metric:<18} {main_val:.4f}     {comp_val:.4f}     {diff:+.4f}")
                else:
                    diff = main_val - comp_val
                    print(f"{metric:<18} {main_val:.4f}     {comp_val:.4f}     {diff:+.4f}")



In [3]:
# Alternative: Direct text input evaluation
def evaluate_text_directly():
    """Evaluate by pasting text directly"""
    print("\n📝 Direct Text Evaluation")
    print("Paste your texts below:")

    ground_truth = input("Paste GROUND TRUTH text:\n")
    extracted = input("Paste EXTRACTED text:\n")

    evaluator = TextExtractionEvaluator()

    # Calculate metrics directly
    print("\n" + "=" * 60)
    print("DIRECT TEXT EVALUATION RESULTS")
    print("=" * 60)

    metrics = evaluator.calculate_metrics(ground_truth, extracted)

    for metric, value in metrics.items():
        if metric in ['WER', 'CER']:
            print(f"{metric:<18}: {value:.4f} ({value*100:.2f}%)")
        elif metric == 'Character_Accuracy':
            print(f"{metric:<18}: {value:.4f} ({value*100:.2f}%)")
        else:
            print(f"{metric:<18}: {value:.4f}")

    # Quality assessment
    print("\n" + "=" * 60)
    print("QUALITY ASSESSMENT")
    print("=" * 60)
    evaluator.quality_assessment(metrics)


In [4]:
# Demo with sample data
def run_demo():
    """Run a demo with sample data"""
    print("\n🎯 RUNNING DEMO WITH SAMPLE DATA")
    print("=" * 50)

    # Sample ground truth
    ground_truth = """
    Part-A
    1. The two most common Supervised task are Classification and Regression.
    2. Validation Dataset is a part of training dataset which is used in cross validation.

    Part-B
    6. For a given dataset, train-test split refers to the phenomenon in which a certain part of dataset is marked as training data.
    Overfitting means when the model performs well on training data but not on testing data.
    """

    # Sample extracted text (with some errors)
    extracted_text = """
    PART-A
    1) The two most common supervised tasks are classification and regression.
    2) Validation dataset is a part of training dataset which is used in cross validation.

    PART-B
    6) For a given dataseet, train-test split refers to the phenomenon in which a certain part of dataset is marked as training data.
    Overfitting means when the model perform well on training data but not on testing data.
    """

    evaluator = TextExtractionEvaluator()

    print("Sample Ground Truth:")
    print(ground_truth[:200] + "...")
    print("\nSample Extracted Text:")
    print(extracted_text[:200] + "...")
    print()

    metrics = evaluator.calculate_metrics(ground_truth, extracted_text)

    print("DEMO RESULTS:")
    print("-" * 40)
    for metric, value in metrics.items():
        if metric in ['WER', 'CER']:
            print(f"{metric:<18}: {value:.4f} ({value*100:.2f}%)")
        elif metric == 'Character_Accuracy':
            print(f"{metric:<18}: {value:.4f} ({value*100:.2f}%)")
        else:
            print(f"{metric:<18}: {value:.4f}")

    print("\nQuality Assessment:")
    evaluator.quality_assessment(metrics)


In [ ]:
import re
from pathlib import Path
from jiwer import wer, cer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import BERTScorer
import nltk
from google.colab import files # Added for upload_and_evaluate

# Assuming TextExtractionEvaluator, upload_and_evaluate, evaluate_text_directly, and run_demo
# are defined in preceding cells or imported. For automatic execution, ensure all cells above
# this one have been run.

def main_menu():
    """Main menu for the evaluation tool"""
    print("\n" + "="*70)
    print("🔍 TEXT EXTRACTION EVALUATION TOOL - GOOGLE COLAB VERSION")
    print("="*70)
    print("\nChoose an option:")
    print("1. 📤 Upload files and evaluate")
    print("2. 📝 Paste text directly for evaluation")
    print("3. 🎯 Run demo with sample data")
    print("4. ℹ️  Show metric explanations")
    print("5. 🚪 Exit")

    choice = input("\nEnter your choice (1-5): ").strip()

    if choice == '1':
        upload_and_evaluate()
    elif choice == '2':
        evaluate_text_directly()
    elif choice == '3':
        run_demo()
    elif choice == '4':
        show_metric_explanations()
    elif choice == '5':
        print("Goodbye! 👋")
        return
    else:
        print("❌ Invalid choice. Please try again.")

    # Return to menu
    input("\nPress Enter to return to main menu...")
    main_menu()

def show_metric_explanations():
    """Show explanations of all metrics"""
    print("\n" + "="*70)
    print("📊 METRIC EXPLANATIONS")
    print("="*70)

    explanations = {
        "WER (Word Error Rate)": "Percentage of incorrect words (insertions, deletions, substitutions). Lower is better.",
        "CER (Character Error Rate)": "Percentage of incorrect characters. Lower is better.",
        "BLEU Score": "Measures n-gram precision similarity to reference text. Higher is better (0-1 scale).",
        "ROUGE-1": "Unigram (single word) overlap between texts. Higher is better.",
        "ROUGE-2": "Bigram (two-word sequence) overlap between texts. Higher is better.",
        "ROUGE-L": "Longest common subsequence similarity. Higher is better.",
        "BERTScore": "Semantic similarity using BERT embeddings. Higher is better (0-1 scale).",
        "Character Accuracy": "Direct character-by-character matching accuracy. Higher is better."
    }

    for metric, explanation in explanations.items():
        print(f"• {metric}:")
        print(f"  {explanation}\n")

    print("\n🎯 QUALITY RATINGS:")
    print("  ✅ EXCELLENT: Professional-grade extraction")
    print("  ✅ GOOD: Suitable for most applications")
    print("  ⚠️  MODERATE: Needs improvement")
    print("  ❌ POOR: Unreliable extraction")

# Start the application
print("\n🚀 Text Extraction Evaluation Tool is ready!")
print("Run main_menu() to start evaluating your text extractions!")

# The automatic call to main_menu() below caused a NameError because
# dependent functions (like upload_and_evaluate) were not yet defined.
# To fix, ensure all preceding cells are run, then call main_menu() manually.
# Uncomment the line below to start automatically ONLY after running all preceding cells:
main_menu()


🚀 Text Extraction Evaluation Tool is ready!
Run main_menu() to start evaluating your text extractions!

🔍 TEXT EXTRACTION EVALUATION TOOL - GOOGLE COLAB VERSION

Choose an option:
1. 📤 Upload files and evaluate
2. 📝 Paste text directly for evaluation
3. 🎯 Run demo with sample data
4. ℹ️  Show metric explanations
5. 🚪 Exit

Enter your choice (1-5): 1
📤 Upload your text files:
1. First upload the GROUND TRUTH file
2. Then upload the EXTRACTED TEXT file
3. (Optional) Upload a COMPARISON file if you want to compare two extractions


Saving 21_v2.docx to 21_v2.docx
Saving 21_combined.txt to 21_combined.txt

📁 Uploaded files: ['21_v2.docx', '21_combined.txt']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.04G [00:00<?, ?B/s]


EVALUATING MAIN EXTRACTION
🔍 Loading files...
📊 Ground truth file: 21_v2.docx
📊 Extracted file: 21_combined.txt
📝 Ground truth length: 136109 characters
📝 Extracted text length: 6548 characters

OVERALL EVALUATION METRICS
WER               : 0.9972 (99.72%)
CER               : 0.9574 (95.74%)
BLEU              : 0.0000
ROUGE-1           : 0.0124
ROUGE-2           : 0.0009
ROUGE-L           : 0.0087
BERTScore         : 0.0000
Character_Accuracy: 0.0037 (0.37%)

SECTION-WISE EVALUATION

QUALITY ASSESSMENT
📈 Extraction Quality:
  ❌ Word Error Rate: POOR
  ❌ Character Error Rate: POOR
  ❌ Semantic Similarity: POOR
  ❌ Character Accuracy: POOR

Press Enter to return to main menu...

🔍 TEXT EXTRACTION EVALUATION TOOL - GOOGLE COLAB VERSION

Choose an option:
1. 📤 Upload files and evaluate
2. 📝 Paste text directly for evaluation
3. 🎯 Run demo with sample data
4. ℹ️  Show metric explanations
5. 🚪 Exit

Enter your choice (1-5): 1
📤 Upload your text files:
1. First upload the GROUND TRUTH file


Saving 21_gt.txt to 21_gt.txt
Saving 21_combined.txt to 21_combined (1).txt

📁 Uploaded files: ['21_gt.txt', '21_combined (1).txt']

EVALUATING MAIN EXTRACTION
🔍 Loading files...
📊 Ground truth file: 21_gt.txt
📊 Extracted file: 21_combined (1).txt
📝 Ground truth length: 4449 characters
📝 Extracted text length: 6548 characters

OVERALL EVALUATION METRICS
WER               : 0.4470 (44.70%)
CER               : 0.4274 (42.74%)
BLEU              : 0.6114
ROUGE-1           : 0.8172
ROUGE-2           : 0.7385
ROUGE-L           : 0.7907
BERTScore         : 0.8002
Character_Accuracy: 0.0570 (5.70%)

SECTION-WISE EVALUATION

📋 PART-A METRICS:
----------------------------------------
  WER             : 0.2128 (21.28%)
  CER             : 0.1667 (16.67%)
  BLEU            : 0.7309
  ROUGE-1         : 0.8955
  ROUGE-2         : 0.8442
  ROUGE-L         : 0.8955
  BERTScore       : 0.9142
  Character_Accuracy: 0.0717 (7.17%)

📋 PART-B METRICS:
----------------------------------------
  WER        